# 參數管理

在選擇了架構並設置了超參數後，我們就進入了訓練階段。
此時，我們的目標是找到使損失函數最小化的模型參數值。
經過訓練後，我們將需要使用這些參數來做出未來的預測。
此外，有時我們希望提取參數，以便在其他環境中復用它們，
將模型保存下來，以便它可以在其他軟體中執行，
或者為了獲得科學的理解而進行檢查。

之前的介紹中，我們只依靠深度學習框架來完成訓練的工作，
而忽略了操作參數的具體細節。
本節，我們將介紹以下內容：

* 訪問參數，用於調試、診斷和視覺化；
* 參數初始化；
* 在不同模型組件間共享參數。

(**我們首先看一下具有單隱藏層的多層感知機。**)


In [1]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4))
net(X)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


tensor([[-0.1726],
        [-0.2085]], grad_fn=<AddmmBackward0>)

## [**參數訪問**]

我們從已有模型中訪問參數。
當通過`Sequential`類定義模型時，
我們可以通過索引來訪問模型的任意層。
這就像模型是一個列表一樣，每層的參數都在其屬性中。
如下所示，我們可以檢查第二個全連接層的參數。


In [2]:
print(net[2].state_dict())

OrderedDict({'weight': tensor([[-0.0621, -0.2993, -0.3190, -0.1819, -0.2301, -0.3458,  0.1106, -0.1286]]), 'bias': tensor([-0.0924])})


輸出的結果告訴我們一些重要的事情：
首先，這個全連接層包含兩個參數，分別是該層的權重和偏置。
兩者都儲存為單精度浮點數（float32）。
注意，參數名稱允許唯一標識每個參數，即使在包含數百個層的網路中也是如此。

### [**目標參數**]

注意，每個參數都表示為參數類的一個實例。
要對參數執行任何操作，首先我們需要訪問底層的數值。
有幾種方法可以做到這一點。有些比較簡單，而另一些則比較通用。
下面的程式碼從第二個全連接層（即第三個神經網路層）提取偏置，
提取後返回的是一個參數類實例，並進一步訪問該參數的值。


In [3]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([-0.0924], requires_grad=True)
tensor([-0.0924])


參數是複合的物件，包含值、梯度和額外資訊。
這就是我們需要顯式參數值的原因。
除了值之外，我們還可以訪問每個參數的梯度。
在上面的這個網路中，由於我們還沒有調用反向傳播，所以參數的梯度處於初始狀態。


In [4]:
net[2].weight.grad == None

True

### [**一次性存取所有參數**]

當我們需要對所有參數執行操作時，逐個訪問它們可能會很麻煩。
當我們處理更複雜的塊（例如，嵌套塊）時，情況可能會變得特別複雜，
因為我們需要遞歸整個樹來提取每個子塊的參數。
下面，我們將通過演示來比較訪問第一個全連接層的參數和訪問所有層。


In [5]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


這為我們提供了另一種訪問網路參數的方式，如下所示。


In [6]:
net.state_dict()['2.bias'].data

tensor([-0.0924])

### [**從嵌套塊收集參數**]

讓我們看看，如果我們將多個塊相互嵌套，參數命名約定是如何工作的。
我們首先定義一個生成塊的函數（可以說是“塊工廠”），然後將這些塊組合到更大的塊中。


In [ ]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在這裡嵌套
        net.add_module(f'block {i}', block1())
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

tensor([[-0.1125],
        [-0.1125]], grad_fn=<AddmmBackward0>)

[**設計了網路後，我們看看它是如何工作的。**]


In [8]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


因為層是分層巢狀的，所以我們也可以像透過巢狀列表索引一樣存取它們。
下面，我們存取第一個主要的區塊中、第二個子區塊的第一層的偏置項。


In [9]:
rgnet[0][1][0].bias.data

tensor([ 0.1710, -0.4831, -0.2082, -0.4978,  0.4221, -0.2434, -0.2803, -0.3249])

## [**參數初始化**]

知道了如何訪問參數後，現在我們看看如何正確地初始化參數。
我們在 :numref:`sec_numerical_stability`中討論了良好初始化的必要性。
深度學習框架提供默認隨機初始化，
也允許我們創建自定義初始化方法，
滿足我們通過其他規則實現初始化權重。


預設情況下，PyTorch會根據一個範圍均勻地初始化權重和偏置矩陣，
這個範圍是根據輸入和輸出維度計算出的。
PyTorch的`nn.init`模組提供了多種預置初始化方法。


### [**內置初始化**]

首先，我們調用內置的初始化器。
下面的程式碼將所有權重參數初始化為標準差為0.01的高斯隨機變量，
且將偏置參數設置為0。


In [10]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([ 0.0009,  0.0026,  0.0083, -0.0088]), tensor(0.))

我們還可以將所有參數初始化為給定的常數，比如初始化為1。


In [11]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

我們還可以[**對某些區塊應用不同的初始化方法**]。
例如，下面我們使用Xavier初始化方法初始化第一個神經網路層，
然後將第三個神經網路層初始化為常數值42。


In [12]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([-0.2620, -0.6892, -0.2948, -0.5449])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


### [**自定義初始化**]

有時，深度學習框架沒有提供我們需要的初始化方法。
在下面的例子中，我們使用以下的分布為任意權重參數$w$定義初始化方法：

$$
\begin{aligned}
    w \sim \begin{cases}
        U(5, 10) & \text{ 可能性 } \frac{1}{4} \\
            0    & \text{ 可能性 } \frac{1}{2} \\
        U(-10, -5) & \text{ 可能性 } \frac{1}{4}
    \end{cases}
\end{aligned}
$$


同樣，我們實現了一個`my_init`函數來應用到`net`。


In [13]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[-7.3321, -7.8969,  9.7363, -5.0903],
        [ 0.0000,  7.6205, -0.0000, -0.0000]], grad_fn=<SliceBackward0>)

注意，我們始終可以直接設置參數。


In [14]:
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]

tensor([42.0000, -6.8969, 10.7363, -4.0903])

## [**參數繫結**]

有時我們希望在多個層間共享參數：
我們可以定義一個稠密層，然後使用它的參數來設置另一個層的參數。


In [15]:
# 我們需要給共享層一個名稱，以便可以引用它的參數
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 1))
net(X)
# 檢查參數是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# 確保它們實際上是同一個物件，而不只是有相同的值
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


這個例子表明第三個和第五個神經網路層的參數是繫結的。
它們不僅值相等，而且由相同的張量表示。
因此，如果我們改變其中一個參數，另一個參數也會改變。
這裡有一個問題：當參數繫結時，梯度會發生什麼情況？
答案是由於模型參數包含梯度，因此在反向傳播期間第二個隱藏層
（即第三個神經網路層）和第三個隱藏層（即第五個神經網路層）的梯度會加在一起。


## 小結

* 我們有幾種方法可以訪問、初始化和繫結模型參數。
* 我們可以使用自定義初始化方法。

## 練習

1. 使用 :numref:`sec_model_construction` 中定義的`FancyMLP`模型，訪問各層的參數。
1. 查看初始化模組文件以了解不同的初始化方法。
1. 構建包含共享參數層的多層感知機並對其進行訓練。在訓練過程中，觀察模型各層的參數和梯度。
1. 為什麼共享參數是個好主意？


[Discussions](https://discuss.d2l.ai/t/1829)


練習一：

1. 使用 :numref:`sec_model_construction` 中定義的`FancyMLP`模型，訪問各層的參數。

我的回答：


讓我們先重新定義 `FancyMLP` 模型，然後訪問其參數：

```python
class FancyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, x):
        x = self.linear(x)
        x = nn.functional.relu(torch.mm(x, self.rand_weight) + 1)
        x = self.linear(x)  # 重複使用全連接層
        while x.abs().sum() > 1:
            x /= 2
        return x.sum()

# 創建模型實例
net = FancyMLP()

# 訪問參數
print("所有命名參數：")
for name, param in net.named_parameters():
    print(f"層名: {name}")
    print(f"參數形狀: {param.shape}")
    print(f"參數值前幾個: {param.data[:2]}\n")

print("訪問特定層：")
print(f"linear層權重形狀: {net.linear.weight.shape}")
print(f"linear層偏置形狀: {net.linear.bias.shape}")
print(f"rand_weight形狀: {net.rand_weight.shape}")
```

這個例子展示了：
1. 如何訪問所有命名參數
2. 如何直接訪問特定層的參數
3. 如何區分需要梯度的參數和不需要梯度的參數（rand_weight）

注意：`linear` 層被重複使用，所以雖然在前向傳播中使用了兩次，但實際上只有一組參數。


練習二：

2. 查看初始化模組文件以了解不同的初始化方法。

我的回答：



PyTorch 的 `nn.init` 模組提供了多種初始化方法，讓我們來看主要的幾種：

````python
import torch
import torch.nn as nn

# 創建一個示例層來展示不同的初始化方法
layer = nn.Linear(4, 3)

# 1. 常用的初始化方法
print("常用初始化方法示例：")

# 正態分布初始化
nn.init.normal_(layer.weight, mean=0, std=0.01)
print("正態分布初始化：\n", layer.weight.data)

# 均勻分布初始化
nn.init.uniform_(layer.weight, a=-0.1, b=0.1)
print("\n均勻分布初始化：\n", layer.weight.data)

# Xavier/Glorot 初始化
nn.init.xavier_uniform_(layer.weight)
print("\nXavier均勻初始化：\n", layer.weight.data)

nn.init.xavier_normal_(layer.weight)
print("\nXavier正態初始化：\n", layer.weight.data)

# Kaiming/He 初始化
nn.init.kaiming_uniform_(layer.weight, mode='fan_in', nonlinearity='relu')
print("\nKaiming均勻初始化：\n", layer.weight.data)

nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')
print("\nKaiming正態初始化：\n", layer.weight.data)

# 2. 常數初始化
nn.init.constant_(layer.bias, 0.)
print("\n常數初始化（偏置）：\n", layer.bias.data)

# 3. 單位矩陣初始化
if layer.weight.shape[0] == layer.weight.shape[1]:  # 只適用於方陣
    nn.init.eye_(layer.weight)
    print("\n單位矩陣初始化：\n", layer.weight.data)

# 4. 正交初始化
nn.init.orthogonal_(layer.weight)
print("\n正交初始化：\n", layer.weight.data)
````

主要初始化方法說明：

1. 基礎初始化：
   - `normal_`: 正態分布初始化
   - `uniform_`: 均勻分布初始化
   - `constant_`: 常數初始化

2. 高級初始化：
   - `xavier_uniform_`/`xavier_normal_`: 適用於 tanh 激活函數
   - `kaiming_uniform_`/`kaiming_normal_`: 適用於 ReLU 激活函數
   - `orthogonal_`: 創建正交矩陣
   - `eye_`: 創建單位矩陣

3. 使用建議：
   - ReLU 網路推薦使用 Kaiming 初始化
   - tanh/sigmoid 網路推薦使用 Xavier 初始化
   - 偏置通常初始化為 0
   - 根據具體任務可能需要調整標準差

4. 自定義初始化示例：
````python
def custom_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.constant_(m.bias, 0)

net = nn.Sequential(nn.Linear(4, 3), nn.ReLU())
net.apply(custom_init)  # 應用到整個網路
````

選擇合適的初始化方法對模型的訓練速度和最終性能都有重要影響。




練習三：

3. 構建包含共享參數層的多層感知機並對其進行訓練。在訓練過程中，觀察模型各層的參數和梯度。

我的回答：





以下是一個包含共享參數層的多層感知機實現和訓練示例：

`````python
import torch
import torch.nn as nn

# 1. 定義模型
class SharedParamMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden = nn.Linear(input_size, hidden_size)  # 共享層
        self.output = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # 使用兩次共享層
        h1 = torch.relu(self.hidden(x))
        h2 = torch.relu(self.hidden(h1))  # 重複使用相同的層
        out = self.output(h2)
        return out

# 2. 創建模型和數據
model = SharedParamMLP(input_size=20, hidden_size=10, output_size=1)
X = torch.randn(100, 20)  # 100個樣本
y = torch.randn(100, 1)   # 隨機標籤

# 3. 訓練設置
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# 4. 訓練循環
print("初始參數：")
for name, param in model.named_parameters():
    print(f"{name}: {param.data[:2]}")

for epoch in range(3):  # 訓練3個epoch作為示例
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    
    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {loss.item():.4f}")
    
    # 觀察參數和梯度
    print("\n參數和梯度：")
    for name, param in model.named_parameters():
        print(f"{name}:")
        print(f"參數值: {param.data[:2]}")
        print(f"梯度值: {param.grad[:2]}")
    
    optimizer.step()

# 5. 驗證參數共享
print("\n驗證參數是否真的共享：")
h1 = model.hidden(X)
h2 = model.hidden(h1)
print("兩次使用是否為同一層：", id(model.hidden) == id(model.hidden))
`````


這個例子展示了：

1. 模型結構：
- 共享的隱藏層被使用了兩次
- 所有參數都被正確追蹤和更新

2. 觀察重點：
- 參數值的變化
- 梯度的累積（共享層的梯度是兩次使用的總和）
- 參數共享的驗證

3. 訓練過程中可以觀察到：
- 共享層的梯度比輸出層大（因為梯度累積）
- 參數更新反映了梯度的累積效果
- loss 的變化顯示模型在學習

4. 注意事項：
- 共享參數可以減少模型參數量
- 但可能使訓練更加困難
- 需要適當調整學習率

這種參數共享的方式在很多現代網絡中都有應用，比如：
- 循環神經網絡（RNN）
- 卷積神經網絡（CNN）
- Transformer 中的自注意力機制


練習四：

4. 為什麼共享參數是個好主意？

我的回答：



共享參數是個好主意，主要有以下原因：

1. 減少參數數量：
- 降低模型複雜度
- 減少內存使用
- 減少過擬合風險

2. 特徵重用：
- 相同的特徵提取器可以在不同位置/時間使用
- 提高模型的泛化能力
- 更好地捕捉數據中的模式

3. 實際應用：
- CNN中的卷積核重用
- RNN中的時間步共享
- Transformer中的自注意力機制

4. 計算效率：
- 更少的參數意味著更快的訓練和推理
- 更容易部署到資源受限的設備

這就是為什麼參數共享在現代深度學習中如此普遍的原因。
